# 01 从一个波长算起：雪粒半径怎样改变反照率

积雪像元变暗，可能来自雪面积减小，也可能来自雪粒变粗。开始解混以前，我想先把后一个因素单独算清楚：**固定纯净球形雪和照明条件，只改变半径，哪些波段最敏感？** 先预测：50 μm 变成 1000 μm 后，可见光和 1 μm 附近是否会下降同样多？把预测记下，再用图检验。

这页先在 1.025 μm 逐项代入 WW80 闭式解，再重算一条光谱和整个网格。输入为官方 SNICAR 202105 库中的单次散射反照率 $\omega$ 和不对称因子 $g$；它们已经包含上游 Mie 散射计算。我们重算的是辐射传输这一步。`ss_alb` 保留 float64，`asm_prm` 存为 float32，计算统一提升至 float64。

第一段代码应打印：`光谱数组：148 半径 × 480 波长 × 3 角度 × 2 光学选项`。本页应出现手算表、单条光谱校核图、四种半径的双角度光谱图，以及太阳谱加权反照率图。来源、转换和逐文件哈希见 `data/demo/scattering_inputs.source.json`。

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowmix import teaching as t
from snowmix.optics import ww80_semi_infinite
forward = t.load_forward()
optics = t.load_optical_inputs()
print("光谱数组：148 半径 × 480 波长 × 3 角度 × 2 光学选项")
print("绘图标签：" + ("中文" if t.configure_plots() else "English（未安装支持的中文字体）"))
assert optics["ss_alb"].shape == optics["asm_prm"].shape == (148, 480, 2)
assert forward["ww_albedo"].shape == (148, 480, 3, 2)
np.testing.assert_allclose(optics["wavelength_um"], forward["wavelength_um"], atol=1e-12)
np.testing.assert_array_equal(optics["radii_um"], forward["radii_um"])

## 先把一个数算明白

取半径 200 μm、Warren 1984、太阳天顶角 60°。缓存网格最接近 1.03 μm 的中心是 **1.025 μm**，这里使用实际中心。$\omega$ 表示一次作用后仍被散射的能量比例；$g$ 越大，前向散射越强。入射角通过 $\mu_0=\cos\theta_0$ 进入模型。

先做 delta 缩放，去除很窄的前向峰：
$$g^*=\frac{g}{1+g},\qquad
\omega^*=\frac{(1-g^2)\omega}{1-g^2\omega}.$$
再定义 $a=1-\omega^*g^*$、$b=g^*/a$、$\xi=\sqrt{3a(1-\omega^*)}$、$P=2\xi/(3a)$。半无限雪层直接入射反照率为
$$\alpha_\infty=\frac{\omega^*}{1+P}\frac{1-b\xi\mu_0}{1+\xi\mu_0}.$$

下面逐行算中间量，随后才调用共享函数。输出是该波长的无量纲方向—半球反照率；应接近 **0.656708**。它把“较强吸收导致较低反照率”的描述落实为可以检查的数值关系。

In [ ]:
ri = int(np.flatnonzero(optics["radii_um"] == 200)[0])
wi = int(np.argmin(abs(optics["wavelength_um"] - 1.03)))
ai = int(np.flatnonzero(forward["angles_deg"] == 60)[0])
omega = float(optics["ss_alb"][ri, wi, 0])
g = float(optics["asm_prm"][ri, wi, 0])
mu0 = float(np.cos(np.deg2rad(60.0)))
g_star = g / (1 + g)
omega_star = (1 - g**2) * omega / (1 - g**2 * omega)
a = 1 - omega_star * g_star
b = g_star / a
xi = np.sqrt(3 * a * (1 - omega_star))
p = 2 * xi / (3 * a)
alpha_manual = omega_star / (1 + p) * (1 - b * xi * mu0) / (1 + xi * mu0)
alpha_function = float(ww80_semi_infinite(omega, g, mu0))
alpha_cache = float(forward["ww_albedo"][ri, wi, ai, 0])
display(pd.DataFrame({"quantity": ["wavelength_um", "omega", "g", "mu0", "g_star", "omega_star", "a", "b", "xi", "P", "manual", "function", "WW80 cache"],
                      "value": [optics["wavelength_um"][wi], omega, g, mu0, g_star, omega_star, a, b, xi, p, alpha_manual, alpha_function, alpha_cache]}))
np.testing.assert_allclose(alpha_manual, alpha_function, atol=1e-12, rtol=0)
assert abs(alpha_function - alpha_cache) < 1e-6
print(f"单波长校核：{optics['wavelength_um'][wi]:.3f} μm，反照率 {alpha_manual:.8f}")

## 从单波长扩展到一条光谱

现在只扩展波长轴：`ss_alb[ri, :, 0]` 是 480 个波长，半径、角度和光学选项仍固定。输出也应为 480 个数。先看曲线是否重合，再看差值数量级；两条曲线在普通坐标上重合时，右侧差值图更容易发现问题。

In [ ]:
spectrum = ww80_semi_infinite(optics["ss_alb"][ri, :, 0], optics["asm_prm"][ri, :, 0], mu0)
reference = forward["ww_albedo"][ri, :, ai, 0]
assert spectrum.shape == (480,)
assert np.max(abs(spectrum - reference)) < 1e-6
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(optics["wavelength_um"], reference, label="MATLAB WW80 cache", color="black", lw=2)
axes[0].plot(optics["wavelength_um"], spectrum, "--", label="NumPy WW80")
axes[1].plot(optics["wavelength_um"], spectrum - reference)
axes[0].set(xlim=(.3, 2.5), ylabel="Albedo (dimensionless)")
axes[1].set(xlim=(.3, 2.5), ylabel="NumPy - cache (dimensionless)")
for ax in axes: ax.set_xlabel("Wavelength / μm")
axes[0].legend(); fig.tight_layout(); display(fig); plt.close(fig)

## 完整网格怎样排轴

批量计算要扩展半径、角度和冰光学选项，采用 `(radius, wavelength, angle, ice_option)` 的顺序。`[:, :, None, :]` 给光学属性插入角度轴；角度余弦用 `(1, 1, 3, 1)` 广播。这样不需要写 148×3×2 个循环。

角度缓存是整数类型。先转 float64 再使用 `deg2rad`，可以避免小整数输入走低精度浮点计算路径。下面校核 **148×480×3×2＝426,240** 个值，预设最大绝对误差为 $10^{-6}$。校核对象是 WW80；SNICAR-AD 使用另一种辐射传输算法，后面单独比较。

In [ ]:
recomputed = t.recompute_ww80(optics, forward["angles_deg"])
error_max = float(np.max(abs(recomputed - forward["ww_albedo"])))
assert recomputed.size == 426240
assert error_max < 1e-6
# solar_flux 轴为 (angle, wavelength)，转置后对每条光谱做通量加权。
flux = forward["solar_flux"].T[None, :, :, None]
broadband = np.sum(recomputed * flux, axis=1) / np.sum(flux, axis=1)
broadband_max = float(np.max(abs(broadband - forward["broadband_ww"])))
assert broadband_max < 1e-6
print(f"完整网格：{recomputed.size:,} 值；光谱最大差 {error_max:.3e}；加权最大差 {broadband_max:.3e}")

## 用图检验最初的预测

代表半径取 50、200、500、1000 μm。50 μm 和 500–1000 μm 分别用于说明较细新雪与较粗老雪的典型差别；实际雪况会形成连续的粒径分布。图中两角度使用同一坐标和颜色，先看 0.5 μm，再看 1.03、1.6 μm 附近。吸收较强的近红外和短波红外通常更能区分粒径。

这里沿用 SNICAR-AD 的完整参考曲线，观察其主要物理趋势；刚才的 WW80 网格用于下一页原文 Fig. 3 的解混实验。两种模型有各自的输出，数据字段名直接标明分支。

In [ ]:
figure = t.plot_optical_spectra(forward)
display(figure); plt.close(figure)

光谱图显示粒径效应随波长变化；因此，把一个可见光波段直接当作粒径指标容易混入覆盖度差异。为了概括总反射能量，还需按入射太阳谱加权：
$$\overline\alpha=\frac{\sum_c F_{\mathrm{sun},c}\alpha_c}{\sum_cF_{\mathrm{sun},c}}.$$
输入光谱是无量纲，通量在分子分母中约去，输出仍为无量纲。左图回答粒径和冰光学选项对宽波段量的影响；右图单独给出 WW80 相对 SNICAR-AD 的差值，单位为反照率百分点。

In [ ]:
figure = t.plot_broadband_albedo(forward)
display(figure); plt.close(figure)
display(t.optics_table(forward).query("radius_um in [50, 200, 500, 1000] and solar_zenith_deg in [0, 60]").round(5))

## 自己改一次，再解释

前面的断言用于固定基线。下面是独立练习，改 `exercise_radius_um` 或 `exercise_angle_deg` 后，不要求曲线保持基线数值。先预测 200→1000 μm 对可见光和 1.025 μm 的影响，再比较打印结果。角度增大时光路与相函数的作用也会改变反照率，必须把角度保持一致后再比较粒径。

In [ ]:
exercise_radius_um = 1000  # 在 30:10:1500 μm 网格内选择。
exercise_angle_deg = 60.0  # 在 [0, 90]° 内选择。
exercise_ri = int(np.flatnonzero(optics["radii_um"] == exercise_radius_um)[0])
exercise_mu = np.cos(np.deg2rad(float(exercise_angle_deg)))
exercise_spectrum = ww80_semi_infinite(optics["ss_alb"][exercise_ri, :, 0], optics["asm_prm"][exercise_ri, :, 0], exercise_mu)
for wavelength in (.5, 1.025, 1.6):
    index = int(np.argmin(abs(optics["wavelength_um"] - wavelength)))
    print(f"{optics['wavelength_um'][index]:.3f} μm: 基线 {spectrum[index]:.4f} → 练习 {exercise_spectrum[index]:.4f}")

同一个雪面积比例可以对应多条不同粒径的雪谱。下一页固定混合比例的真值，检查把这些谱都近似为 200 μm 时，最小二乘会怎样改变雪系数。

下一步：[02 从三波段手算到 Fig. 3](02_fig3_sma.ipynb)。物理来源为 Wiscombe–Warren (1980)、SNICAR-ADv3 (Flanner et al., 2021) 及随仓库保留的 SPIRES 参考实现；散射参数子集从官方公开库提取并标明精度转换。